# External Validation on NHANES 2021-2023

In [1]:
import pandas as pd
import numpy as np
import os, sys, json, warnings
warnings.filterwarnings("ignore")

def find_project_root(marker="data"):
    """Walk up from CWD until we find a folder containing 'data' and 'src'."""
    here = os.path.abspath(os.getcwd())
    for _ in range(5):
        if os.path.isdir(os.path.join(here, marker)) and os.path.isdir(os.path.join(here, "src")):
            return here
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent
    raise RuntimeError("Could not locate project root")

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

RAW       = os.path.join(PROJECT_ROOT, "data", "raw")
INTERIM   = os.path.join(PROJECT_ROOT, "data", "interim")
PROCESSED = os.path.join(PROJECT_ROOT, "data", "processed")
os.makedirs(INTERIM, exist_ok=True)
os.makedirs(PROCESSED, exist_ok=True)

RANDOM_STATE = 42

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib

print("Project root :", PROJECT_ROOT)
print("RAW          :", RAW)
print("INTERIM      :", INTERIM)
print("PROCESSED    :", PROCESSED)
print("RAW exists   :", os.path.isdir(RAW))
print("RAW contents :", sorted(os.listdir(RAW)) if os.path.isdir(RAW) else "N/A")

In [2]:
TRAINING_CYCLES = [
    ("NHANES_2003_2004", "_C", "2003-2004", 0),
    ("NHANES_2005_2006", "_D", "2005-2006", 1),
    ("NHANES_2007_2008", "_E", "2007-2008", 2),
    ("NHANES_2009_2010", "_F", "2009-2010", 3),
    ("NHANES_2011_2012", "_G", "2011-2012", 4),
    ("NHANES_2013_2014", "_H", "2013-2014", 5),
    ("NHANES_2015_2016", "_I", "2015-2016", 6),
    ("NHANES_2017_2018", "_J", "2017-2018", 7),
]
EXTERNAL_CYCLE = ("NHANES_2021_2023", "_L", "2021-2023", 8)

COMPONENTS = ["DEMO","BMX","GHB","GLU","BPX","BIOPRO","DIQ","KIQ_U",
              "SMQ","ALQ","PAQ","DBQ","MCQ","BPQ","CBC","HDL",
              "TCHOL","TRIGLY","INS","HSCRP","ALB_CR","HSQ"]

FILE_OVERRIDE = {
    ("BPX","_L"): "BPXO_L.xpt",
    ("HSCRP","_C"): "CRP_C.xpt",
    ("HSCRP","_D"): "CRP_D.xpt",
    ("HSCRP","_E"): "CRP_E.xpt",
    ("HSCRP","_F"): "CRP_F.xpt",
    ("HSCRP","_G"): "CRP_G.xpt",
}

REQUIRED_COLS = {
    "DEMO":   ["SEQN","RIDSTATR","RIDAGEYR","RIAGENDR","RIDRETH3","INDFMPIR",
               "DMDEDUC2","DMDMARTL","RIDEXPRG","DMDBORN4","DMDYRSUS",
               "INDHHIN2","DMDHHSIZ","RIDEXMON","WTMEC2YR","SDMVPSU","SDMVSTRA"],
    "BMX":    ["SEQN","BMXWT","BMXHT","BMXBMI","BMXWAIST","BMXHIP",
               "BMXARMC","BMXARML","BMXLEG"],
    "GHB":    ["SEQN","LBXGH"],
    "GLU":    ["SEQN","LBXGLU","WTSAF2YR"],
    "BPX":    ["SEQN","BPXSY1","BPXDI1","BPXSY2","BPXDI2","BPXSY3","BPXDI3",
               "BPXSY4","BPXDI4","BPXPLS","BPXPULS"],
    "BIOPRO": ["SEQN","LBXSCR","LBXSBU","LBXSAL","LBXSTP","LBXSCA","LBXSNASI",
               "LBXSKSI","LBXSCLSI","LBXSTB","LBXSAPSI","LBXSATSI","LBXSASSI",
               "LBXSGTSI","LBXSPH","LBXSC3SI","LBXSUA","LBXSIR","LBXSGB",
               "LBXSCK","LBXSLDSI","LBXSOSSI"],
    "DIQ":    ["SEQN","DIQ010","DIQ160","DID040","DIQ080"],
    "KIQ_U":  ["SEQN","KIQ022","KIQ025","KIQ026"],
    "SMQ":    ["SEQN","SMQ020","SMD030","SMQ040","SMD650","SMD057",
               "SMQ890","SMQ900","SMQ910"],
    "ALQ":    ["SEQN","ALQ111","ALQ121","ALQ130","ALQ142","ALQ151"],
    "PAQ":    ["SEQN","PAQ605","PAQ620","PAQ635","PAQ650","PAQ665","PAD680"],
    "DBQ":    ["SEQN","DBQ700","DBQ197","DBD895","DBD900","DBD905","DBD910"],
    "MCQ":    ["SEQN","MCQ160B","MCQ160C","MCQ160D","MCQ160E","MCQ160F",
               "MCQ160L","MCQ300C","MCQ300A"],
    "BPQ":    ["SEQN","BPQ020"],
    "CBC":    ["SEQN","LBXWBCSI","LBXHGB","LBXRDW","LBXPLTSI","LBXLYPCT","LBXNEPCT"],
    "HDL":    ["SEQN","LBDHDD"],
    "TCHOL":  ["SEQN","LBXTC"],
    "TRIGLY": ["SEQN","LBXTR"],
    "INS":    ["SEQN","LBXIN"],
    "HSCRP":  ["SEQN","LBXHSCRP"],
    "ALB_CR": ["SEQN","URDACT"],
    "HSQ":    ["SEQN","HSD010"],
}

C:\Users\MiR\AppData\Local\Temp\ipykernel_6420\4253358614.py:68: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_master["ASCVD_risk"] = train_master.apply(ascvd_risk, axis=1)
C:\Users\MiR\AppData\Local\Temp\ipykernel_6420\4253358614.py:83: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_master["met_syndrome_count"] = train_master.apply(met_syndrome_count, axis=1)
C:\Users\MiR\AppData\Local\Temp\ipykernel_6420\4253358614.py:94: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fra

Training medians computed for 110 features


In [3]:
def harmonise(df, suffix):
    if "LBXCRP" in df.columns and "LBXHSCRP" not in df.columns:
        df = df.rename(columns={"LBXCRP": "LBXHSCRP"})

    if suffix in ["_C","_D","_E","_F","_G","_H","_I"]:
        alq_map = {}
        if "ALQ101"  in df.columns: alq_map["ALQ101"]  = "ALQ111"
        if "ALQ120Q" in df.columns: alq_map["ALQ120Q"] = "ALQ121"
        if "ALQ141Q" in df.columns: alq_map["ALQ141Q"] = "ALQ142"
        if alq_map:
            df = df.rename(columns=alq_map)

    if suffix == "_L":
        bp_map = {
            "BPXOSY1":"BPXSY1","BPXODI1":"BPXDI1",
            "BPXOSY2":"BPXSY2","BPXODI2":"BPXDI2",
            "BPXOSY3":"BPXSY3","BPXODI3":"BPXDI3",
            "BPXOSY4":"BPXSY4","BPXODI4":"BPXDI4",
            "BPXOPLS":"BPXPLS","BPXOPULS":"BPXPULS",
        }
        df = df.rename(columns={k:v for k,v in bp_map.items() if k in df.columns})
        if "LBXTLG" in df.columns and "LBXTR" not in df.columns:
            df = df.rename(columns={"LBXTLG": "LBXTR"})

    return df

Loaded DEMO: (11933, 11), columns=11
Loaded BMX: (8860, 10), columns=10
Loaded GHB: (7199, 3), columns=3
Loaded GLU: (3996, 3), columns=3
Loaded BPX: (7801, 8), columns=8
Loaded BIOPRO: (7199, 23), columns=23
Loaded DIQ: (11744, 5), columns=5
Loaded KIQ_U: (7809, 4), columns=4
Loaded SMQ: (9015, 5), columns=5
Loaded ALQ: (6337, 7), columns=7
Loaded PAQ: (8153, 3), columns=3
Loaded DBQ: (11933, 2), columns=2
Loaded MCQ: (11744, 8), columns=8
Loaded BPQ: (8501, 3), columns=3
Loaded CBC: (8727, 8), columns=8
Loaded HDL: (8068, 3), columns=3
Loaded TCHOL: (8068, 3), columns=3
Loaded TRIGLY: (3996, 3), columns=3
Loaded INS: (3996, 3), columns=3
Loaded HSCRP: (8727, 3), columns=3
Loaded ALB_CR: (8493, 3), columns=3
Loaded HSQ: (6615, 2), columns=2
Merged BMX: (8860, 19)
Merged GHB: (7199, 20)
Merged GLU: (3996, 21)
Merged BPX: (3996, 27)
Merged BIOPRO: (3996, 48)
Merged DIQ: (3996, 51)
Merged KIQ_U: (3425, 53)
Merged SMQ: (3425, 56)
Merged ALQ: (3425, 61)
Merged PAQ: (3425, 62)
Merged DBQ: (

In [4]:
missing_report = {}
loaded_per_cycle = {}

for folder, suffix, label, code in TRAINING_CYCLES + [EXTERNAL_CYCLE]:
    folder_path = os.path.join(RAW, folder)
    if not os.path.isdir(folder_path):
        print(f"❌ {folder} not found, skipping.")
        continue

    print(f"\n=== {label} ({suffix}) ===")
    cycle_frames = {}

    for comp in COMPONENTS:
        fname = FILE_OVERRIDE.get((comp, suffix), f"{comp}{suffix}.xpt")
        fpath = os.path.join(folder_path, fname)
        if not os.path.exists(fpath):
            print(f"  {comp}: file missing ({fname})")
            continue

        df = pd.read_sas(fpath)
        df = harmonise(df, suffix)
        df["cycle"] = code

        keep = [c for c in REQUIRED_COLS[comp] if c in df.columns]
        df = df[keep].copy()

        missing = [c for c in REQUIRED_COLS[comp] if c not in df.columns]
        if missing:
            missing_report[(label, comp)] = missing

        cycle_frames[comp] = df

    loaded_per_cycle[label] = cycle_frames
    print(f"  Loaded {len(cycle_frames)} components")

print(f"\n=== Missing variable report ===")
if missing_report:
    for (lab, comp), miss in missing_report.items():
        print(f"{lab} {comp}: {miss}")
else:
    print("No missing variables across any component.")

print("\nAll cycles loaded.")

All derived features computed on external data.


In [5]:
training_frames = []

for folder, suffix, label, code in TRAINING_CYCLES:
    if label not in loaded_per_cycle:
        continue
    cycle_frames = loaded_per_cycle[label]
    if "DEMO" not in cycle_frames:
        print(f"{label}: DEMO missing, skipping")
        continue

    merged = cycle_frames["DEMO"].copy()
    for comp in COMPONENTS:
        if comp == "DEMO" or comp not in cycle_frames:
            continue
        right = cycle_frames[comp]
        merged = merged.merge(right, on=["SEQN", "cycle"], how="inner")
    training_frames.append(merged)
    print(f"{label}: merged {merged.shape}")

if not training_frames:
    raise RuntimeError("No training frames loaded — check paths in Cell 2")

train_master = pd.concat(training_frames, ignore_index=True, sort=False)
print(f"\nCombined training shape: {train_master.shape}")
print("Duplicate SEQN+cycle:", train_master[["SEQN", "cycle"]].duplicated().sum())

Evaluating diabetes...
Saved risk scores for diabetes
Evaluating hypertension...


C:\Users\MiR\AppData\Local\Temp\ipykernel_6420\3216404862.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_external[col] = merged[col]
c:\Mir\OneHealth-ML\.venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\MiR\AppData\Local\Temp\ipykernel_6420\3216404862.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_external[col] = merged[col]
c:\Mir\OneHealth-ML\.venv\Lib\site-packages\sklearn\utils\validation.py

Saved risk scores for hypertension
Evaluating ckd...
Saved risk scores for ckd
Evaluating dyslipidemia...
Saved risk scores for dyslipidemia
Evaluating cvd...
Saved risk scores for cvd
Evaluating metabolic_syndrome...


C:\Users\MiR\AppData\Local\Temp\ipykernel_6420\3216404862.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_external[col] = merged[col]
c:\Mir\OneHealth-ML\.venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\MiR\AppData\Local\Temp\ipykernel_6420\3216404862.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_external[col] = np.nan
C:\Users\MiR\AppData\Local\Temp\ipykernel_6420\3216404862.py:32: Performance

Saved risk scores for metabolic_syndrome
Evaluating liver_disease...
Saved risk scores for liver_disease
External predictions saved to ../data/processed\external_predictions_2021_2023.csv


c:\Mir\OneHealth-ML\.venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\MiR\AppData\Local\Temp\ipykernel_6420\3216404862.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_external[col] = merged[col]
c:\Mir\OneHealth-ML\.venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [6]:
external_frames = loaded_per_cycle.get(EXTERNAL_CYCLE[2], {})
if external_frames and "DEMO" in external_frames:
    ext = external_frames["DEMO"].copy()
    for comp in COMPONENTS:
        if comp == "DEMO" or comp not in external_frames:
            continue
        ext = ext.merge(external_frames[comp], on=["SEQN", "cycle"], how="inner")
    print(f"External master shape: {ext.shape}")
else:
    ext = None
    print("External cycle not available.")

diabetes: AUROC=0.903, AUPRC=0.795, F1=0.681, n_pos=666, n_neg=2731
hypertension: AUROC=0.878, AUPRC=0.913, F1=0.818, n_pos=1935, n_neg=1490
ckd: AUROC=0.873, AUPRC=0.575, F1=0.462, n_pos=307, n_neg=3118
dyslipidemia: AUROC=0.856, AUPRC=0.911, F1=0.767, n_pos=1836, n_neg=1265
cvd: AUROC=0.830, AUPRC=0.424, F1=0.430, n_pos=434, n_neg=2991
metabolic_syndrome: AUROC=0.846, AUPRC=0.703, F1=0.688, n_pos=957, n_neg=1935
liver_disease: AUROC=0.843, AUPRC=0.439, F1=0.421, n_pos=425, n_neg=2990

External validation metrics saved.
